In [1]:
import pandas as pd
import numpy as np
import os



In [2]:
np.random.seed(42)



In [3]:
train_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv"
test_path = "/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv"
sample_sub_path = (
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_sub_path)



In [4]:
# Compute a simple linear regression (slope) of FVC vs Weeks on the whole training set
weeks = train["Weeks"].values
fvc_vals = train["FVC"].values
# variance of weeks; add a tiny epsilon to avoid division by zero in pathological cases
var_weeks = np.var(weeks) if np.var(weeks) != 0 else 1e-8
slope = np.cov(weeks, fvc_vals, bias=True)[0, 1] / var_weeks

# Overall mean FVC kept as a fallback for any missing patient baseline
overall_mean_fvc = train["FVC"].mean()

# Map each patient in the test set to its baseline FVC (Week 0 measurement)
baseline_dict = test.set_index("Patient")["FVC"].to_dict()


def predict_fvc(row):
    """Predict FVC for a given Patient_Week using baseline + linear trend."""
    patient_week = row["Patient_Week"]
    # Patient_Week format: <PatientID>_<WeekNumber>
    patient_id, week_str = patient_week.rsplit("_", 1)
    week = int(week_str)
    baseline = baseline_dict.get(patient_id, overall_mean_fvc)
    return baseline + slope * week


submission = sample_submission.copy()
submission["FVC"] = submission.apply(predict_fvc, axis=1)
# Use a high confidence value; after clipping at 70 this helps the metric
submission["Confidence"] = 1000.0



In [5]:
submission = submission[["Patient_Week", "FVC", "Confidence"]]
submission.to_csv("submission.csv", index=False)



In [6]:
print("Submission file created with shape:", submission.shape)
print("First rows:")
print(submission.head())

Submission file created with shape: (1908, 3)
First rows:
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2376.689446      1000.0
1  ID00126637202218610655908_-2  2376.126298      1000.0
2  ID00126637202218610655908_-1  2375.563149      1000.0
3   ID00126637202218610655908_0  2375.000000      1000.0
4   ID00126637202218610655908_1  2374.436851      1000.0
